# Baseline evaluation — LFM2.5-Audio-1.5B (EN)Weekend plan **steps 1 & 2**: score the *vanilla* model before any fine-tuning.| Campaign | Question set | Scorers ||---|---|---|| A — audio quality | `benchmark/baseline_en/questions.jsonl` (24 open questions) | WER (re-transcription), DNSMOS, NISQA || B — tool calling | `benchmark/toolcalling_en/cases.sample.jsonl` (12 cases) | tool_call (BFCL-style) |Runtime: **L4 (22 GB)**. Reports are written to `reports/` and downloaded at the end.

In [ ]:
# Clone the repo and install (liquid backend only: no vLLM needed for a baseline).import os, subprocess, sysREPO_URL = "https://github.com/rcarvalo/finetuning_s2s_toolcalling"BRANCH   = "rd/pr_rca_eval_baseline"WORK     = "/content/finetuning_s2s_toolcalling"if not os.path.exists(WORK):    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=False)os.chdir(WORK)# Colab gotcha: always install through the *kernel's* interpreter.subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[serving-liquid,eval]"], check=True)print("install OK")

In [ ]:
# HF login (gated/private checkpoints) and sanity-check the GPU.from huggingface_hub import loginlogin()import torchassert torch.cuda.is_available(), "select an L4 runtime (Runtime > Change runtime type)"print(torch.cuda.get_device_name(0))

In [ ]:
# DNSMOS weights (sig_bak_ovr.onnx) are not redistributable via PyPI: fetch once.import urllib.request, pathlibDNSMOS_DIR = pathlib.Path("/content/dnsmos"); DNSMOS_DIR.mkdir(exist_ok=True)DNSMOS_PATH = DNSMOS_DIR / "sig_bak_ovr.onnx"if not DNSMOS_PATH.exists():    urllib.request.urlretrieve(        "https://github.com/microsoft/DNS-Challenge/raw/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx",        DNSMOS_PATH,    )os.environ["DNSMOS_MODEL_PATH"] = str(DNSMOS_PATH)# NISQA weights are optional: without NISQA_MODEL_PATH the scorer reports# `unavailable` and the campaign continues (WER + DNSMOS still cover step 1).print("DNSMOS ready:", DNSMOS_PATH.stat().st_size // 1024, "KiB")

In [ ]:
# Which metrics can actually run on this machine?subprocess.run([sys.executable, "-m", "lfm2_audio.cli.eval.suite", "--list-scorers"], check=True)

In [ ]:
# Campaign A — audio quality of spoken answers (step 1).CHECKPOINT = "LiquidAI/LFM2.5-Audio-1.5B"os.makedirs("reports", exist_ok=True)subprocess.run([sys.executable, "-m", "lfm2_audio.cli.eval.suite",    "--checkpoint", CHECKPOINT, "--backend", "liquid",    "--questions", "benchmark/baseline_en/questions.jsonl",    "--scorers", "wer,dnsmos,nisqa",    "--max-tokens", "400",    "--out", "reports/baseline_en_audio.json"], check=True)

In [ ]:
# Campaign B — tool-calling decisions (step 2).subprocess.run([sys.executable, "-m", "lfm2_audio.cli.eval.suite",    "--checkpoint", CHECKPOINT, "--backend", "liquid",    "--questions", "benchmark/toolcalling_en/cases.sample.jsonl",    "--scorers", "tool_call",    "--max-tokens", "200",    "--out", "reports/baseline_en_toolcalling.json"], check=True)

In [ ]:
# Summary — the numbers every later fine-tune will be compared against.import jsonfor name in ("baseline_en_audio", "baseline_en_toolcalling"):    report = json.load(open(f"reports/{name}.json"))    print(f"=== {name} ===")    print(json.dumps({k: report[k] for k in ("context", "cases", "metrics") if k in report}, indent=2)[:2000])

In [ ]:
# Keep the reports: download them, then commit them to the branch from your machine.from google.colab import filesfor name in ("baseline_en_audio", "baseline_en_toolcalling"):    files.download(f"reports/{name}.json")

In [ ]:
# Step 3 (bonus while the GPU is warm): inventory the Rcarvalo datasets.
# Requires the HF login cell above; writes docs/dataset_inventory.md.
subprocess.run([sys.executable, "-m", "lfm2_audio.cli.data.inventory",
    "--author", "Rcarvalo", "--out", "docs/dataset_inventory.md"], check=True)
files.download("docs/dataset_inventory.md")